# LSTM hyperparameter search (Colab, GPU)

Bayesian search (Optuna TPE sampler + median pruner) over the LSTM from `model_LSTM/lstm.py`.
Self-contained: only the feature CSVs written by `features.ipynb` are needed.

**Before running:** *Runtime → Change runtime type → GPU* (e.g. T4). Then *Runtime → Run all*.

The 52 validation engines are split in half by engine: the **tune** half scores every trial,
the **holdout** half is scored once by the final model, so its RMSE was not used to pick
the hyperparameters. Test engines are only used for the final predictions.

In [ ]:
!pip install -q optuna

import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")
if DEVICE.type != "cuda":
    print("WARNING: no GPU. Runtime -> Change runtime type -> GPU, then rerun.")

## Data

Clones the repo to get `Features/`. If the repo is private, the clone fails and you are asked
to upload the four CSVs from your local `Features/` folder instead.

In [ ]:
import os, shutil
from google.colab import files

REPO = "https://github.com/stef-ann/RUL-engine.git"
NEEDED = ["features_train.csv", "features_val.csv", "features_test.csv"]

if not all(os.path.exists(f"Features/{f}") for f in NEEDED):
    os.system(f"git clone -q --depth 1 {REPO} repo")
    if os.path.isdir("repo/Features"):
        shutil.copytree("repo/Features", "Features", dirs_exist_ok=True)
    else:
        print("Clone failed (private repo?). Upload the CSVs from Features/:")
        os.makedirs("Features", exist_ok=True)
        for name, data in files.upload().items():
            open(f"Features/{name}", "wb").write(data)

print(sorted(os.listdir("Features")))

## Where to save the study (optional)

Colab wipes `/content` when the runtime disconnects. Set `USE_DRIVE = True` to keep the
Optuna database in Google Drive, so rerunning the search cell continues the same study.

In [ ]:
USE_DRIVE = False

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs("/content/drive/MyDrive/RUL", exist_ok=True)
    STORAGE = "sqlite:////content/drive/MyDrive/RUL/optuna_lstm.db"
else:
    STORAGE = "sqlite:///optuna_lstm.db"
print(STORAGE)

## Settings

In [ ]:
N_TRIALS = 50          # Optuna trials
TIMEOUT = None         # seconds, or None for no limit
MAX_EPOCHS = 30        # per trial; the best epoch is kept

FEATURES = ["Ps30", "T50", "BPR", "htBleed", "T24", "T30",
            "health", "health_delta", "health_slope", "Cycle"]
RUL_CAP = 125          # piecewise-linear target: RUL is flat early in life

DEFAULTS = dict(window=30, hidden=32, layers=1, dropout=0.0, lr=1e-3,
                batch=256, epochs=MAX_EPOCHS, l1=0.0, l2=0.0, seed=0)

## Data preparation

In [ ]:
import numpy as np
import pandas as pd

train = pd.read_csv("Features/features_train.csv")
val = pd.read_csv("Features/features_val.csv")
test = pd.read_csv("Features/features_test.csv")

# Standardize with training statistics only.
mu, sd = train[FEATURES].mean(), train[FEATURES].std()
for df in (train, val, test):
    df[FEATURES] = (df[FEATURES] - mu) / sd

# Split validation engines: alternate engines go to tune / holdout.
val_ids = sorted(val["EngineID"].unique())
val_tune = val[val["EngineID"].isin(val_ids[0::2])]
val_hold = val[val["EngineID"].isin(val_ids[1::2])]


def make_windows(df, window, with_target=True, last_only=False):
    """One window ending at every flight of every engine.

    Each engine is left-padded by repeating its first row, so every flight
    (even on the shortest test engine) ends a full-length window.
    """
    X, y, ids = [], [], []
    for eid, g in df.groupby("EngineID", sort=True):
        g = g.sort_values("Cycle")
        feats = g[FEATURES].to_numpy(np.float32)
        feats = np.vstack([np.repeat(feats[:1], window - 1, axis=0), feats])
        rows = [len(g) - 1] if last_only else range(len(g))
        for r in rows:
            X.append(feats[r:r + window])
            ids.append(eid)
        if with_target:
            rul = np.minimum(g["RUL"].to_numpy(np.float32), RUL_CAP)
            y.extend(rul[list(rows)])
    X = np.stack(X)
    y = np.array(y, np.float32) if with_target else None
    return X, y, np.array(ids)


def to_gpu(X, y=None):
    X = torch.from_numpy(X).to(DEVICE)
    return X if y is None else (X, torch.from_numpy(y).to(DEVICE))


_cache = {}
def windows(w):
    """Train and tune-engine windows for window length w, kept on the GPU."""
    if w not in _cache:
        X_tr, y_tr, _ = make_windows(train, w)
        X_va, y_va, _ = make_windows(val_tune, w)
        _cache[w] = (*to_gpu(X_tr, y_tr), *to_gpu(X_va, y_va))
    return _cache[w]

print("train rows", len(train), "| tune engines", len(val_ids[0::2]),
      "| holdout engines", len(val_ids[1::2]))

## Model and training

In [ ]:
from torch import nn

torch.backends.cudnn.benchmark = True


class LSTMRegressor(nn.Module):
    def __init__(self, n_features, hidden, layers=1, dropout=0.0):
        super().__init__()
        self.lstm = nn.LSTM(n_features, hidden, num_layers=layers, batch_first=True,
                            dropout=dropout if layers > 1 else 0.0)
        self.drop = nn.Dropout(dropout)
        self.head = nn.Linear(hidden, 1)

    def forward(self, x):                      # x: (batch, window, n_features)
        out, _ = self.lstm(x)
        return self.head(self.drop(out[:, -1])).squeeze(-1)   # last step only


def penalty(model, l1, l2):
    """Lasso + ridge over weight matrices only (biases are not shrunk)."""
    weights = [p for n, p in model.named_parameters() if "weight" in n]
    return (l1 * sum(w.abs().sum() for w in weights)
            + l2 * sum(w.pow(2).sum() for w in weights))


@torch.no_grad()
def predict(model, X, chunk=8192):
    """RUL predictions in flights, as a NumPy array."""
    model.eval()
    out = [model(X[i:i + chunk]) for i in range(0, len(X), chunk)]
    return (torch.cat(out) * RUL_CAP).cpu().numpy()


def rmse(pred, true):
    true = true.cpu().numpy() if torch.is_tensor(true) else true
    return float(np.sqrt(np.mean((pred - true) ** 2)))


def train_model(cfg, X_tr, y_tr, X_va, y_va, on_epoch=None):
    """Train one LSTM with all data on the GPU. Returns model, val RMSE per epoch."""
    torch.manual_seed(cfg["seed"])
    model = LSTMRegressor(X_tr.shape[2], cfg["hidden"], cfg["layers"],
                          cfg["dropout"]).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg["lr"])
    y_scaled = y_tr / RUL_CAP
    n, history = len(X_tr), []
    for epoch in range(1, cfg["epochs"] + 1):
        model.train()
        perm = torch.randperm(n, device=DEVICE)
        for i in range(0, n, cfg["batch"]):
            idx = perm[i:i + cfg["batch"]]
            optimizer.zero_grad(set_to_none=True)
            loss = nn.functional.mse_loss(model(X_tr[idx]), y_scaled[idx])
            loss = loss + penalty(model, cfg["l1"], cfg["l2"])
            loss.backward()
            optimizer.step()
        history.append(rmse(predict(model, X_va), y_va))
        if on_epoch is not None:
            on_epoch(epoch, history[-1])
    return model, history


# Quick check: default settings, a few epochs.
import time
t0 = time.time()
_, h = train_model(dict(DEFAULTS, epochs=3), *windows(DEFAULTS["window"]))
print(f"3 epochs in {time.time() - t0:.1f}s, tune-engine RMSE per epoch:",
      [round(v, 2) for v in h])

## Bayesian search

Rerunning this cell adds more trials to the same study (stored at `STORAGE`).

In [ ]:
import optuna


def objective(trial):
    cfg = dict(DEFAULTS)
    cfg.update(
        window=trial.suggest_int("window", 10, 50, step=10),
        hidden=trial.suggest_int("hidden", 16, 128, log=True),
        layers=trial.suggest_int("layers", 1, 2),
        dropout=trial.suggest_float("dropout", 0.0, 0.5),
        lr=trial.suggest_float("lr", 1e-4, 1e-2, log=True),
        batch=trial.suggest_categorical("batch", [128, 256, 512]),
        l1=trial.suggest_float("l1", 1e-9, 1e-4, log=True),
        l2=trial.suggest_float("l2", 1e-9, 1e-3, log=True),
    )

    def on_epoch(epoch, val_rmse):
        trial.report(val_rmse, epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()

    _, history = train_model(cfg, *windows(cfg["window"]), on_epoch=on_epoch)
    best_epoch = int(np.argmin(history)) + 1
    trial.set_user_attr("best_epoch", best_epoch)
    return history[best_epoch - 1]


study = optuna.create_study(
    study_name="lstm_rul", storage=STORAGE, load_if_exists=True,
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=0),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=5),
)
study.optimize(objective, n_trials=N_TRIALS, timeout=TIMEOUT)

## Results

In [ ]:
states = optuna.trial.TrialState
done = [t for t in study.trials if t.state == states.COMPLETE]
pruned = [t for t in study.trials if t.state == states.PRUNED]
print(f"{len(done)} trials completed, {len(pruned)} pruned")

best = study.best_trial
print(f"best tune-engine RMSE {best.value:.2f} "
      f"(trial {best.number}, epoch {best.user_attrs['best_epoch']})")
for k, v in best.params.items():
    print(f"  {k:8s} {v:.3g}" if isinstance(v, float) else f"  {k:8s} {v}")

study.trials_dataframe().sort_values("value").head(10)

In [ ]:
from optuna.visualization import plot_optimization_history, plot_param_importances

plot_optimization_history(study).show()
plot_param_importances(study).show()

## Refit the best setting, score the holdout engines, predict test

The command line for running the same setting locally with `model_LSTM/lstm.py` is printed too.

In [ ]:
cfg = dict(DEFAULTS, **best.params, epochs=best.user_attrs["best_epoch"])
model, _ = train_model(cfg, *windows(cfg["window"]))
w = cfg["window"]

X_ho, y_ho, _ = make_windows(val_hold, w)
X_hl, y_hl, _ = make_windows(val_hold, w, last_only=True)
print(f"holdout RMSE, all windows      : {rmse(predict(model, to_gpu(X_ho)), y_ho):.2f}")
print(f"holdout RMSE, last flight only : {rmse(predict(model, to_gpu(X_hl)), y_hl):.2f}")

X_te, _, te_ids = make_windows(test, w, with_target=False, last_only=True)
preds = pd.DataFrame({"EngineID": te_ids,
                      "RUL_pred": predict(model, to_gpu(X_te)).round(1)})
preds.to_csv("lstm_tuned_test_predictions.csv", index=False)

print("\nlocal equivalent:\npython model_LSTM/lstm.py " + " ".join(
    f"--{k} {v:.3g}" if isinstance(v, float) else f"--{k} {v}"
    for k, v in cfg.items() if k != "seed"))

In [ ]:
# Download the predictions (and the study database, if not stored in Drive).
files.download("lstm_tuned_test_predictions.csv")
if not USE_DRIVE:
    files.download("optuna_lstm.db")